In [21]:
import pandas as pd
import numpy as np
import pickle

# Load model
with open('data/model.pkl', 'rb') as f:
    model = pickle.load(f)

# Load features (now includes referee and raw columns)
df = pd.read_csv('data/features.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Referee stats calculator (uses data already in CSV)
def get_referee_stats(referee, before_date):
    past = df[(df['Referee'] == referee) & (df['Date'] < before_date)]
    if len(past) == 0:
        return None
    return {
        'ref_matches':       len(past),
        'ref_home_win_rate': (past['target'] == 0).mean(),
        'ref_yellows_avg':   (past['HY'] + past['AY']).mean(),
        'ref_reds_avg':      (past['HR'] + past['AR']).mean(),
        'ref_goals_avg':     (past['FTHG'] + past['FTAG']).mean(),
    }

print(f"✅ Data loaded: {len(df)} matches")
print(f"✅ Unique referees: {df['Referee'].nunique()}")

✅ Data loaded: 4004 matches
✅ Unique referees: 45


In [22]:
FEATURE_COLS = [
    'home_form_last5',
    'away_form_last5',
    'home_avg_scored',
    'home_avg_conceded',
    'away_avg_scored',
    'away_avg_conceded',
    'ref_matches',
    'ref_home_win_rate',
    'ref_yellows_avg',
    'ref_reds_avg',
    'ref_goals_avg',
    'elo_home',
    'elo_away',
    'elo_diff',
    'h2h_matches',
    'h2h_home_wins',
    'h2h_draws',
    'h2h_goals_avg',
]

def predict_match(home_team, away_team, referee=None):
    # Get latest features for each team
    home_data = df[df['HomeTeam'] == home_team].tail(1)
    away_data = df[df['AwayTeam'] == away_team].tail(1)

    if home_data.empty:
        print(f"❌ Team not found: {home_team}")
        return
    if away_data.empty:
        print(f"❌ Team not found: {away_team}")
        return

    # Default referee stats (league averages)
    ref_features = {
        'ref_matches':       0,
        'ref_home_win_rate': 0.45,
        'ref_yellows_avg':   3.5,
        'ref_reds_avg':      0.1,
        'ref_goals_avg':     2.7,
    }

    # Override with real referee stats if provided
    if referee:
        stats = get_referee_stats(referee, pd.Timestamp.now())
        if stats:
            ref_features = stats
            print(f"🟨 Referee: {referee} ({stats['ref_matches']} matches in database)")
        else:
            print(f"⚠️ Referee {referee} not found, using league averages")

    # H2H stats between these two teams
    h2h_past = df[
        ((df['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)) |
        ((df['HomeTeam'] == away_team) & (df['AwayTeam'] == home_team))
    ].tail(5)

    if len(h2h_past) == 0:
        h2h_features = {
            'h2h_matches':   0,
            'h2h_home_wins': 0.33,
            'h2h_draws':     0.25,
            'h2h_goals_avg': 2.7,
        }
    else:
        home_wins = sum(
            1 for _, p in h2h_past.iterrows()
            if (p['HomeTeam'] == home_team and p['target'] == 0) or
               (p['HomeTeam'] == away_team and p['target'] == 2)
        )
        draws = (h2h_past['target'] == 1).sum()
        h2h_features = {
            'h2h_matches':   len(h2h_past),
            'h2h_home_wins': home_wins / len(h2h_past),
            'h2h_draws':     draws / len(h2h_past),
            'h2h_goals_avg': (h2h_past['FTHG'] + h2h_past['FTAG']).mean(),
        }

    features = {
        'home_form_last5':   home_data['home_form_last5'].values[0],
        'away_form_last5':   away_data['away_form_last5'].values[0],
        'home_avg_scored':   home_data['home_avg_scored'].values[0],
        'home_avg_conceded': home_data['home_avg_conceded'].values[0],
        'away_avg_scored':   away_data['away_avg_scored'].values[0],
        'away_avg_conceded': away_data['away_avg_conceded'].values[0],
        **ref_features,
        'elo_home':          home_data['elo_home'].values[0],
        'elo_away':          away_data['elo_away'].values[0],
        'elo_diff':          home_data['elo_home'].values[0] - away_data['elo_away'].values[0],
        **h2h_features,
    }

    X = np.array([[features[f] for f in FEATURE_COLS]])
    probs = model.predict_proba(X)[0]

    print(f"\n⚽ {home_team} vs {away_team}")
    print(f"─────────────────────────────")
    print(f"🏠 {home_team} wins:  {probs[0]:.1%}")
    print(f"🤝 Draw:              {probs[1]:.1%}")
    print(f"✈️  {away_team} wins: {probs[2]:.1%}")
    print(f"─────────────────────────────")

    max_idx = np.argmax(probs)
    if max_idx == 0:
        prediction = f"{home_team} wins"
    elif max_idx == 1:
        prediction = "Draw"
    else:
        prediction = f"{away_team} wins"

    print(f"📊 Prediction: {prediction}")
    return probs

# Test
predict_match("Arsenal", "Man City")
print()
predict_match("Arsenal", "Man City", referee="M Oliver")


⚽ Arsenal vs Man City
─────────────────────────────
🏠 Arsenal wins:  56.9%
🤝 Draw:              21.5%
✈️  Man City wins: 21.6%
─────────────────────────────
📊 Prediction: Arsenal wins

🟨 Referee: M Oliver (307 matches in database)

⚽ Arsenal vs Man City
─────────────────────────────
🏠 Arsenal wins:  44.1%
🤝 Draw:              28.6%
✈️  Man City wins: 27.3%
─────────────────────────────
📊 Prediction: Arsenal wins


array([0.44095703, 0.28623861, 0.27280436])

In [23]:
# Show all available teams
teams = sorted(set(df['HomeTeam'].unique()) | set(df['AwayTeam'].unique()))
print(f"Total teams: {len(teams)}\n")
for t in teams:
    print(t)

Total teams: 35

Arsenal
Aston Villa
Bournemouth
Brentford
Brighton
Burnley
Cardiff
Chelsea
Crystal Palace
Everton
Fulham
Huddersfield
Hull
Ipswich
Leeds
Leicester
Liverpool
Luton
Man City
Man United
Middlesbrough
Newcastle
Norwich
Nott'm Forest
QPR
Sheffield United
Southampton
Stoke
Sunderland
Swansea
Tottenham
Watford
West Brom
West Ham
Wolves


In [24]:
predict_match("Liverpool", "Chelsea")
print()
predict_match("Man United", "Tottenham")
print()
predict_match("Newcastle", "Aston Villa", referee="A Taylor")


⚽ Liverpool vs Chelsea
─────────────────────────────
🏠 Liverpool wins:  43.6%
🤝 Draw:              25.8%
✈️  Chelsea wins: 30.6%
─────────────────────────────
📊 Prediction: Liverpool wins


⚽ Man United vs Tottenham
─────────────────────────────
🏠 Man United wins:  63.7%
🤝 Draw:              21.1%
✈️  Tottenham wins: 15.2%
─────────────────────────────
📊 Prediction: Man United wins

🟨 Referee: A Taylor (313 matches in database)

⚽ Newcastle vs Aston Villa
─────────────────────────────
🏠 Newcastle wins:  31.7%
🤝 Draw:              22.1%
✈️  Aston Villa wins: 46.2%
─────────────────────────────
📊 Prediction: Aston Villa wins


array([0.31687539, 0.22112562, 0.46199899])